# Instrument Detection - Model Training & Evaluation

This notebook demonstrates model training, hyperparameter tuning, and evaluation.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path('..').resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, f1_score, classification_report, 
                             confusion_matrix, ConfusionMatrixDisplay)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
import joblib

from src.config_loader import config
from src.data_preparation import load_prepared_data
from src.model_trainer import ModelTrainer, prepare_data_split

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

In [ ]:
# Load data
X, y = load_prepared_data()
classes = config.get('classes')

# Train-test split
X_train, X_test, y_train, y_test = prepare_data_split(X, y)

print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples: {X_test.shape[0]}")
print(f"Features: {X_train.shape[1]}")
print(f"Classes: {classes}")

In [ ]:
# Train all models
trainer = ModelTrainer()
results = trainer.train_all_models(X_train, y_train, X_test, y_test)

In [ ]:
# Display results
results_df = pd.DataFrame(trainer.results).T
results_df = results_df[['accuracy', 'f1_macro', 'f1_weighted', 'precision_macro', 'recall_macro']]
results_df = results_df.sort_values('accuracy', ascending=False)

print("Model Comparison (Test Set):")
display(results_df.style.format('{:.4f}').background_gradient(cmap='RdYlGn'))

In [ ]:
# Cross-validation results
cv_data = []
for name, cv_res in trainer.cv_results.items():
    for metric, scores in cv_res.items():
        cv_data.append({
            'model': name,
            'metric': metric,
            'mean': scores.mean(),
            'std': scores.std()
        })

cv_df = pd.DataFrame(cv_data)
cv_pivot = cv_df.pivot(index='model', columns='metric', values='mean')
cv_pivot = cv_pivot[['accuracy', 'f1_macro', 'f1_weighted', 'precision_macro', 'recall_macro']]

print("Cross-Validation Results (Mean):")
display(cv_pivot.style.format('{:.4f}').background_gradient(cmap='RdYlGn'))

In [ ]:
# Select best model
best_name = trainer.select_best_model(X_test, y_test)

# Detailed classification report
print("\nDetailed Classification Report:")
print(trainer.get_classification_report(X_test, y_test))

In [ ]:
# Confusion Matrix
trainer.plot_confusion_matrix(X_test, y_test)

In [ ]:
# Feature Importance (for tree-based models)
trainer.plot_feature_importance()

In [ ]:
# Hyperparameter Tuning for Best Model
print(f"Tuning hyperparameters for {best_name}...")
trainer.hyperparameter_tune(best_name, X_train, y_train)

# Re-evaluate after tuning
tuned_metrics = trainer.evaluate_model(best_name, X_test, y_test)
print(f"\nAfter tuning:")
print(f"  Accuracy: {tuned_metrics['accuracy']:.4f}")
print(f"  F1 (macro): {tuned_metrics['f1_macro']:.4f}")

In [ ]:
# Save the best model
model_dir = Path(config.get('output.model_dir', 'models'))
model_dir.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(model_dir / "best_model.joblib"))

# Verify saved model
trainer2 = ModelTrainer()
trainer2.load_model(str(model_dir / "best_model.joblib"))

# Test prediction
sample_idx = 0
sample_features = X_test[sample_idx:sample_idx+1]
prediction = trainer2.predict(sample_features)
proba = trainer2.predict_proba(sample_features)

print(f"\nTest Prediction:")
print(f"  True label: {classes[y_test[sample_idx]]}")
print(f"  Predicted: {prediction[0]}")
print(f"  Probabilities: {dict(zip(classes, proba[0]))}")